# FCA–MoE F2-E — Nested parametric bootstrap

This notebook installs the F2-E implementation from the research branch, validates the code, runs a small smoke pilot, and provides the publication-oriented command. The pilot is **not** inferential evidence.

The experiment uses the existing Qwen1.5-MoE-A2.7B layer-0 traces (4 sequences × 256 tokens, 60 experts, Top-4). It repeats pairwise-MaxEnt fitting, lambda selection, and temporal Gibbs-weight selection inside each outer bootstrap replicate.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess
REPO = Path('/content/FCA-MoE')
BRANCH = 'research/fca-moe-v0.3-f2e-nested-bootstrap'
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/msellamiTN/FCA-MoE.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
%cd /content/FCA-MoE/research/v03
!python -m pip install -e .
!python -m pip install pytest


In [ ]:
# Unit tests include the exact transition-likelihood validator; implementation tests are not empirical inference.
!python -m pytest -q tests/test_f2e_nested_bootstrap.py tests/test_f2e_temporal_loo_validation.py


In [ ]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/FCA_MoE_v03/qwen15_moe_a27_wikitext_test/contexts_v1')
required = [ROOT/'qwen_contexts_by_layer.npz', ROOT/'sequence_alignment.csv']
for p in required:
    print(p, p.exists())
    if not p.exists(): raise FileNotFoundError(p)
OUT_BASE = ROOT/'sequence_replication/layer_00/temporal/f2e_nested_bootstrap'
OUT_BASE.mkdir(parents=True, exist_ok=True)


## 1. Leave-one-sequence-out temporal validation

Run this before the confirmatory bootstrap. Each fold selects lambda and temporal weights on three sequences only, then evaluates exact conditional transition log likelihood and lag-profile prediction on the held-out sequence. The four-fold diagnostic is not a hypothesis test. F2-E blocks the 999-replicate confirmatory run unless the predeclared diagnostic passes.


In [ ]:
LOO_OUT = ROOT/'sequence_replication/layer_00/temporal/loo_confirmatory'
LOO_OUT.mkdir(parents=True, exist_ok=True)
!python scripts/f2e_temporal_loo_validation.py --root "{ROOT}" --output "{LOO_OUT}" --lambdas 0.0003 0.001 0.003 --maxiter 400 --weight-grid-step 0.1 --screen-replicates 8 --predictive-replicates 200 --seed 20261120

import json
print(json.dumps(json.loads((LOO_OUT/'loo_validation_summary.json').read_text()), indent=2))


## 1. Smoke pilot

This run checks the end-to-end data path, fitting, conditional indexes, weight selection, FCA mining, and checkpoint output. It uses a coarse ten-point temporal-weight grid, one screening replicate, two outer replicates, and ten inner replicates. **Do not use its p-value as a research result.**


In [ ]:
!python scripts/f2e_nested_bootstrap.py \
  --root "{ROOT}" \
  --output "{OUT_BASE}/pilot" \
  --outer-replicates 2 \
  --inner-replicates 10 \
  --screen-replicates 1 \
  --weight-grid-step 0.5 \
  --lambdas 0.001 \
  --maxiter 250 \
  --checkpoint-every 1 \
  --resume


## 2. Diagnostic run

Only proceed if the smoke pilot completes with zero failed outer replicates and the fitted-model diagnostics are acceptable. This run is still a diagnostic, not the final significance analysis.


In [ ]:
!python scripts/f2e_nested_bootstrap.py \
  --root "{ROOT}" \
  --output "{OUT_BASE}/diagnostic" \
  --outer-replicates 20 \
  --inner-replicates 30 \
  --screen-replicates 2 \
  --weight-grid-step 0.5 \
  --lambdas 0.0003 0.001 0.003 \
  --maxiter 400 \
  --checkpoint-every 1 \
  --resume


## 4. Confirmatory nested bootstrap

The confirmatory run uses the complete six-metric family, 100 inner replicates, 999 outer replicates, 286 temporal candidates and eight screening replicates. It requires the completed leave-one-sequence-out temporal validation summary. This is a major compute job; the notebook does not auto-launch it. Run only after reviewing the LOO output and accepting the predeclared diagnostic. If interrupted, rerun with exactly the same arguments and `--resume`.


In [ ]:
# Confirmatory run — manually uncomment only after reviewing the LOO diagnostics.
# !python scripts/f2e_nested_bootstrap.py --root "{ROOT}" --output "{OUT_BASE}/f2e_confirmatory_v1" --temporal-loo-summary "{LOO_OUT}/loo_validation_summary.json" --outer-replicates 999 --inner-replicates 100 --screen-replicates 8 --weight-grid-step 0.1 --lambdas 0.0003 0.001 0.003 --maxiter 400 --checkpoint-every 1 --resume


In [ ]:
import json
OUT = OUT_BASE/'f2e_confirmatory_v1'
summary = OUT/'f2e_nested_bootstrap_summary.json'
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))
else:
    print('No completed F2-E summary yet. Check the checkpoint CSV and rerun with identical arguments and --resume.')


## Interpretation guardrails

- A successful smoke test validates software plumbing only.
- The inferential result is conditional on the fitted pairwise-temporal null family.
- Four source sequences do not justify broad corpus generalization.
- A successful bootstrap does not establish semantic expert specialization, causality, or inference-speed gains.
- Report failed replicate counts and the exact configuration alongside every p-value.
